In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(67123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.linear()],
        edge=rd.Op.weight(),
        output=rd.Op.linear(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .alters(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-05-16T16:46:05.793755Z  INFO Epoch 1    | Score:   2.0038 | Time: 2.85ms
2026-05-16T16:46:05.793928Z  INFO Epoch 2    | Score:   2.0038 | Time: 2.92ms
2026-05-16T16:46:05.794021Z  INFO Epoch 3    | Score:   2.0038 | Time: 2.98ms
2026-05-16T16:46:05.794090Z  INFO Epoch 4    | Score:   1.6821 | Time: 3.02ms
2026-05-16T16:46:05.794170Z  INFO Epoch 5    | Score:   1.6821 | Time: 3.07ms
2026-05-16T16:46:05.794250Z  INFO Epoch 6    | Score:   1.6821 | Time: 3.12ms
2026-05-16T16:46:05.794337Z  INFO Epoch 7    | Score:   1.6821 | Time: 3.18ms
2026-05-16T16:46:05.794419Z  INFO Epoch 8    | Score:   1.6821 | Time: 3.24ms
2026-05-16T16:46:05.794501Z  INFO Epoch 9    | Score:   1.6821 | Time: 3.29ms
2026-05-16T16:46:05.794585Z  INFO Epoch 10   | Score:   1.6821 | Time: 3.34ms
2026-05-16T16:46:05.794675Z  INFO Epoch 11   | Score:   1.6821 | Time: 3.41ms
2026-05-16T16:46:05.794757Z  INFO Epoch 12   | Score:   1.6821 | Time: 3.47ms
2026-05-16T16:46:05.794833Z  INFO Epoch 13   | Score:   1.6821 |

In [4]:
# collector.plot()

In [5]:
eval_results = result.value().eval(inputs)
accuracy = rd.accuracy(result.value(), inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.66%
	R² Score: 0.99976
	RMSE: 0.03118
	Loss (MSE): 0.00097
})

In [6]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",8.0,108.0,54.0,65.053825,4232.0,NaN,8.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000294,0.001565,0.000783,0.000691,4.7686e-7,NaN,0.000294,0.001271,2,1565µs,782µs,690µs,294µs,1270µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000007,0.000007,0.000007,0.0,0.0,NaN,0.000007,0.000007,1,6µs,6µs,0µs,6µs,6µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""time""",0.000404,0.065628,0.00025,0.000221,4.8798e-8,0.0,0.000039,0.002848,262,65628µs,250µs,220µs,38µs,2847µs,0µs,261,1,"[""time""]"
"""index""",262.0,262.0,262.0,0.0,0.0,NaN,262.0,262.0,1,null,null,null,null,null,null,0,1,"[""statistic""]"
"""score.improvement""",1.0,114.0,1.0,0.0,0.0,0.0,1.0,1.0,114,null,null,null,null,null,null,261,1,"[""statistic"", ""score""]"


In [9]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
3
96
102
103
…
255
257
258


In [ ]:
display(HTML(filtered._repr_html_()))

version
i64
0
3
326
327
328
…
603
605
607
